In [10]:
import csv

def ler_transacoes(caminho_arquivo):
    transacoes = []

    try:
        with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
            leitor = csv.DictReader(arquivo)

            for linha in leitor:
                transacoes.append(linha)

        print(f"Arquivo lido com sucesso. Total de registros: {len(transacoes)}")
        return transacoes

    except FileNotFoundError:
        print("Erro: arquivo não encontrado.")
        return []

In [11]:
from datetime import datetime

def validar_id(linha):
#validar ID (inteiro)
    try:
        if not linha['id'] or not linha['id'].isdigit():
            return False
    except KeyError:
        print("A chave especificada não foi encontrada.")
        return False

    if not linha['cliente_id'] or linha['cliente_id'].strip() == "":
        return False

    return True


def validar_data(linha):
#alidar data (AAAA-MM-DD)
    try:
        datetime.strptime(linha['data'], "%Y-%m-%d")
    except ValueError:
        return False

    if linha['tipo'] not in ["credito", "debito"]:
        return False

    return True

def validar_valor(linha):
    #valor (> 0)
    try:
        valor = float(linha['valor'])
        if valor <= 0:
            return False
    except ValueError:
        return False

    return True

def validar_transacao(linha):
    return validar_id(linha) and validar_data(linha) and validar_valor(linha)


def limpar_dados(dados):
    validos = []
    invalidos = 0

    for linha in dados:
        if validar_transacao(linha):
            validos.append(linha)
        else:
            invalidos += 1

    print("Resumo da validação:")
    print(f"Total de linhas lidas: {len(dados)}")
    print(f"Linhas válidas: {len(validos)}")
    print(f"Linhas inválidas: {invalidos}")
    print("")

    return validos

In [12]:
from datetime import datetime

def processar_datas(dados):
    datas_convertidas = []

    for linha in dados:
        #string para datetime
        data_obj = datetime.strptime(linha['data'], "%Y-%m-%d")
        linha['data_obj'] = data_obj

        #extrair mês
        linha['mes'] = data_obj.strftime("%Y-%m")

        datas_convertidas.append(data_obj)

    #calculo intervalo entre datas
    data_mais_antiga = min(datas_convertidas)
    data_mais_recente = max(datas_convertidas)

    diferenca_dias = (data_mais_recente - data_mais_antiga).days

    print("Resumo de datas:")
    print(f"Data mais antiga: {data_mais_antiga.date()}")
    print(f"Data mais recente: {data_mais_recente.date()}")
    print(f"Diferença em dias: {diferenca_dias}")

    return dados

In [13]:
LIMITE_SUSPEITO = 10000.00
def obter_suspeitas(dados):
    lista = []

    for linha in dados:
        valor = float(linha['valor'])

        if valor > LIMITE_SUSPEITO:
            lista.append({
                "id": int(linha['id']),
                "cliente_id": linha['cliente_id'],
                "data": linha['data'],
                "valor": valor
            })

    return lista

In [14]:
def gerar_relatorio(dados, total_invalidas):
    relatorio = {}

    for linha in dados:
        mes = linha['mes']
        valor = float(linha['valor'])
        tipo = linha['tipo']

        if mes not in relatorio:
            relatorio[mes] = {
                'quantidade': 0,
                'total_credito': 0.0,
                'total_debito': 0.0,
                'valores': []
            }

        relatorio[mes]['quantidade'] += 1
        relatorio[mes]['valores'].append(valor)

        if tipo == 'credito':
            relatorio[mes]['total_credito'] += valor
        else:
            relatorio[mes]['total_debito'] += valor

    # calcular métricas finais
    for mes in relatorio:
        dados_mes = relatorio[mes]
        valores = dados_mes['valores']

        dados_mes['saldo'] = dados_mes['total_credito'] - dados_mes['total_debito']
        dados_mes['media'] = sum(valores) / len(valores)
        dados_mes['maior_valor'] = max(valores)
        dados_mes['menor_valor'] = min(valores)

        del dados_mes['valores']  # remove campo auxiliar

    return relatorio

In [15]:
import json
from datetime import datetime

def salvar_json(dados, total_invalidas):
    estrutura = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(dados),
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": gerar_relatorio(dados, total_invalidas),
        "transacoes_suspeitas": obter_suspeitas(dados)
    }

    with open("relatorio.json", "w", encoding="utf-8") as arquivo:
        json.dump(estrutura, arquivo, ensure_ascii=False, indent=2)

    print("Arquivo relatorio.json gerado com sucesso.")

In [16]:
def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def exibir_relatorio(dados, dados_limpos):
    total_lidas = len(dados)
    total_validas = len(dados_limpos)
    total_invalidas = total_lidas - total_validas

    if not dados_limpos:
        print("Nenhum dado válido para exibir.")
        return

    # Período analisado
    datas = [linha['data_obj'] for linha in dados_limpos]
    data_min = min(datas).date()
    data_max = max(datas).date()

    print("\n==============================")
    print("     RELATÓRIO GERAL")
    print("==============================\n")

    print(f"Período analisado: {data_min} → {data_max}")
    print(f"Total de transações válidas: {total_validas}")
    print(f"Total de transações inválidas: {total_invalidas}")

    print("\n==============================")
    print("     RESUMO MENSAL")
    print("==============================\n")

    #logica de agrupamento
    relatorio = gerar_relatorio(dados_limpos, total_invalidas)

    for mes in sorted(relatorio.keys()):
        r = relatorio[mes]

        print(f"Mês: {mes}")
        print(f"  Transações: {r['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(r['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(r['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(r['saldo'])}")
        print(f"  Média:         {formatar_moeda(r['media'])}")
        print(f"  Maior valor:   {formatar_moeda(r['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(r['menor_valor'])}")
        print()

    print("==============================")
    print("  TRANSAÇÕES SUSPEITAS")
    print("==============================")

    suspeitas = obter_suspeitas(dados_limpos)

    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for s in suspeitas:
            print(
                f"ID: {s['id']} | "
                f"Cliente: {s['cliente_id']} | "
                f"Data: {s['data']} | "
                f"Valor: {formatar_moeda(s['valor'])}"
            )

In [17]:
import numpy as np
import matplotlib.pyplot as plt

def gerar_grafico(resumo_mensal):
    meses = []
    creditos = []
    debitos = []

    for mes, dados in resumo_mensal.items():
        meses.append(mes)
        creditos.append(dados["total_credito"])
        debitos.append(dados["total_debito"])

    # Criação do gráfico
    plt.figure()

    plt.bar(meses, creditos, label="Crédito")
    plt.bar(meses, debitos, bottom=creditos, label="Débito")

    # Títulos e rótulos
    plt.title("Movimentação Financeira Mensal")
    plt.xlabel("Mês")
    plt.ylabel("Valor (R$)")
    plt.legend()

    # Salvar imagem
    plt.savefig("grafico.png")

    # Fechar figura para liberar memória
    plt.close()

In [18]:
# 1 Ler dados
CAMINHO_ARQUIVO = "transacoes.csv"
dados = ler_transacoes(CAMINHO_ARQUIVO)

# 2 Validar dados
dados_validos = []
total_invalidas = 0

for linha in dados:
    if validar_transacao(linha):
        dados_validos.append(linha)
    else:
        total_invalidas += 1

# 3 Processar dados
dados_processados = processar_datas(dados_validos)

# 4 Gerar relatório
relatorio = gerar_relatorio(dados_processados, total_invalidas)

# 5 Salvar relatorio
salvar_json(dados_processados, total_invalidas)

# 6 Exibir no terminal
exibir_relatorio(dados, dados_processados)

# 7 Gerar gráfico
gerar_grafico(relatorio)

Arquivo lido com sucesso. Total de registros: 21
Resumo de datas:
Data mais antiga: 2026-01-05
Data mais recente: 2026-03-28
Diferença em dias: 82
Arquivo relatorio.json gerado com sucesso.

     RELATÓRIO GERAL

Período analisado: 2026-01-05 → 2026-03-28
Total de transações válidas: 15
Total de transações inválidas: 6

     RESUMO MENSAL

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 406,00
  Saldo:         R$ 4.294,00
  Média:         R$ 1.021,20
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 45,00

Mês: 2026-02
  Transações: 4
  Total crédito: R$ 15.000,00
  Total débito:  R$ 529,90
  Saldo:         R$ 14.470,10
  Média:         R$ 3.882,47
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 89,90

Mês: 2026-03
  Transações: 6
  Total crédito: R$ 10.100,00
  Total débito:  R$ 11.449,90
  Saldo:         R$ -1.349,90
  Média:         R$ 3.591,65
  Maior valor:   R$ 11.000,00
  Menor valor:   R$ 99,90

  TRANSAÇÕES SUSPEITAS
ID: 5 | Cliente: CLI003 | Dat